In [17]:
%%writefile ling_features.py
import stanza
import os

def init_stanza():
    if not os.path.exists("/root/.cache/stanza"):
        stanza.download("uk")

    nlp = stanza.Pipeline(
        lang="uk",
        processors="tokenize,mwt,pos,lemma",
        use_gpu=False
    )
    return nlp


def extract_ling_features(text, nlp):

    if not isinstance(text, str):
        text = str(text)

    doc = nlp(text)

    lemmas = []
    pos_tags = []

    for sent in doc.sentences:
        for word in sent.words:
            lemmas.append(word.lemma)
            pos_tags.append(word.upos)

    return {
        "lemma_text": " ".join(lemmas),
        "pos_seq": pos_tags
    }


def filter_pos(lemmas, pos_tags, allowed=("NOUN", "ADJ")):
    filtered = [
        lemma for lemma, pos in zip(lemmas, pos_tags)
        if pos in allowed
    ]
    return " ".join(filtered)

Overwriting ling_features.py


In [1]:
!pip install stanza scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 20.4 MB/s eta 0:00:00


In [32]:
import pandas as pd

path = "/content/processed_v2.csv"
df = pd.read_csv(path)

print("Dataset:", df.shape)

USE_SUBSET = True
SUBSET_SIZE = 500

if USE_SUBSET:
    df = df[df['label'].isin(["Real", "Fake"])]
    df = df.sample(SUBSET_SIZE, random_state=42)

print("Dataset:", df.shape)
df.head()

Dataset: (12956, 2)
Dataset: (500, 2)


,processed_text,label
5322,прем'єрка італії мелоні вимагає відшкодування ...,Real
7457,хто більше витрачає на оборону — китай чи сша ...,Real
5751,"білий дім заявив, що авдіївка під загрозою зах...",Real
3519,"у харкові вибухи, повідомляє мер терехов у хар...",Real
457,я вчора відвідав карпати і був шокований тим щ...,Fake


In [33]:
df["label"].value_counts()

,count
label,
Real,290
Fake,210


In [34]:
print(df.columns)

df = df.rename(columns={"processed_text": "text"})

print(df.columns)
df.head()

Index(['processed_text', 'label'], dtype='object')
Index(['text', 'label'], dtype='object')


,text,label
5322,прем'єрка італії мелоні вимагає відшкодування ...,Real
7457,хто більше витрачає на оборону — китай чи сша ...,Real
5751,"білий дім заявив, що авдіївка під загрозою зах...",Real
3519,"у харкові вибухи, повідомляє мер терехов у хар...",Real
457,я вчора відвідав карпати і був шокований тим щ...,Fake


In [35]:
for i in range(10):
    print("="*60)
    print("RAW:", df.iloc[i]["text"][:300])

RAW: прем'єрка італії мелоні вимагає відшкодування збитків у €100 тис. прем'єрка італії мелоні вимагає відшкодування збитків у €100 тис. через фейкові порновідео з її обличчям відомо, що мелоні буде свідчити в суді італійського міста сассарі 2 липня щодо порноликів, для створення яких був використаний de
RAW: хто більше витрачає на оборону — китай чи сша 1 травня, 14:11 зараз буду розвінчувати міф, який чомусь вліз дуже глибоко в розум багатьох «людей доброї волі» допомагає мені мій улюблений американський економічний блогер ноа сміт. отже, ось перший графік. на ньому показано в доларах військові витрати
RAW: білий дім заявив, що авдіївка під загрозою захоплення російськими військами. пояснюють це відсутністю артилерійських снарядів у українських військових. білий дім заявив, що авдіївка під загрозою захоплення російськими військами. пояснюють це відсутністю артилерійських снарядів у українських військов
RAW: у харкові вибухи, повідомляє мер терехов у харкові вибухи, повідомляє мер тер

In [36]:
from ling_features import init_stanza, extract_ling_features, filter_pos

df["text"] = df["text"].fillna("").astype(str)

nlp = init_stanza()

lemma_texts = []
pos_filtered_texts = []

for i, text in enumerate(df["text"]):
    if i % 50 == 0:
        print(f"Processing {i}/{len(df)}")

    result = extract_ling_features(text, nlp)

    lemma_texts.append(result["lemma_text"])

    filtered = filter_pos(
        result["lemma_text"].split(),
        result["pos_seq"]
    )

    pos_filtered_texts.append(filtered)

df["lemma_text"] = lemma_texts
df["pos_filtered_text"] = pos_filtered_texts

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: uk (Ukrainian) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/uk/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: uk (Ukrainian):
| Processor | Package     |
---------------------------
| tokenize  | iu          |
| mwt       | iu          |
| pos       | iu_charlm   |
| lemma     | iu_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


Processing 0/500
Processing 50/500
Processing 100/500
Processing 150/500
Processing 200/500
Processing 250/500
Processing 300/500
Processing 350/500
Processing 400/500
Processing 450/500


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def run_baseline(text_column):

    X_train, X_test, y_train, y_test = train_test_split(
        df[text_column],
        df["label"],
        test_size=0.2,
        random_state=42,
        stratify=df["label"]
    )

    vectorizer = TfidfVectorizer(max_features=20000)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=200)
    model.fit(X_train_tfidf, y_train)

    preds = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")

    return acc, f1


acc_raw, f1_raw = run_baseline("text")
acc_lemma, f1_lemma = run_baseline("lemma_text")
acc_pos, f1_pos = run_baseline("pos_filtered_text")

print("Processed_v2:", acc_raw, f1_raw)
print("Lemma:", acc_lemma, f1_lemma)
print("Lemma + POS filter:", acc_pos, f1_pos)

Processed_v2: 0.7 0.6357455075279261
Lemma: 0.73 0.6871741397288842
Lemma + POS filter: 0.71 0.6442154336891179


In [38]:
for i in range(10):
    print("="*60)
    print("RAW:", df.iloc[i]["text"][:300])
    print("LEMMA:", df.iloc[i]["lemma_text"][:300])
    print("POS FILTERED:", df.iloc[i]["pos_filtered_text"][:300])

RAW: прем'єрка італії мелоні вимагає відшкодування збитків у €100 тис. прем'єрка італії мелоні вимагає відшкодування збитків у €100 тис. через фейкові порновідео з її обличчям відомо, що мелоні буде свідчити в суді італійського міста сассарі 2 липня щодо порноликів, для створення яких був використаний de
LEMMA: прем’єрка італія мелонь вимагати відшкодування збиток у € 100 тис. . прем’єрка італія мелонь вимагати відшкодування збиток у € 100 тис. . через фейковий порновідео з її обличчя відомо , що мелонь бути свідчити в суд італійський місто сассар 2 липень щодо порнолик , для створення який бути використан
POS FILTERED: прем’єрка італія мелонь відшкодування збиток € прем’єрка італія мелонь відшкодування збиток € фейковий порновідео обличчя мелонь суд італійський місто 2 липень порнолик створення використаний тіло актриса обличчя італійський прем’єрка слідство 40-річний чоловік фейковий порновідео 73-річний батько с
RAW: хто більше витрачає на оборону — китай чи сша 1 травня, 14:11 зара

In [39]:
import os

os.makedirs("docs", exist_ok=True)

with open("docs/audit_summary_lab3.md", "w", encoding="utf-8") as f:
    f.write("# Lab3 Audit Summary\n\n")
    f.write(f"Raw Accuracy: {acc_raw}\n")
    f.write(f"Lemma Accuracy: {acc_lemma}\n")
    f.write(f"Lemma + POS Accuracy: {acc_pos}\n\n")
    f.write("Conclusion:\n")
    f.write("Lemmatization improved performance. POS filtering reduced performance.\n")

print("Audit file created successfully!")

Audit file created successfully!
